# M9 · Maintain useful evidence as software changes

Coverage observes execution; assertions establish particular behavior. Run a controlled defect experiment in temporary files and inspect the actual test and coverage reports.

**Start:** Before class, from `teaching-snippets/`, run `uv sync --extra dev` and select its Python environment as the notebook kernel. Notebook-editor/kernel setup is separate classroom preparation. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

**Actual checks:** The labeled pytest cells run a subprocess using the selected kernel’s interpreter and visible temporary source files. They print real test output and exit status when you execute them. No repository tests or production code are changed. Deliberate failures are handled or captured so the supplied sequence can continue to its recovery.

## Exploration 1 — build and observe the mechanism

### 1. Start with a meaningful behavioral promise

A test checks particular inputs and outcomes. State the shop’s quantity rule and independently expected arithmetic.

**Before running:** Predict the result or state change.

In [ ]:
def calculate_line_total(price_paise, quantity):
    if type(quantity) is not int or quantity <= 0:
        raise ValueError("Quantity must be a positive integer")
    return price_paise * quantity

assert calculate_line_total(49900, 2) == 99800
print(calculate_line_total(49900, 2))


**Change and explain:** Which alternative arithmetic operation would this assertion reject?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Two cables total 99800.

</details>

### 2. Observe a weak comparison surviving a defect

An assertion can stay green when it checks a property the bug preserves.

**Before running:** Predict the result or state change.

In [ ]:
def broken_total(price_paise, quantity):
    return price_paise + quantity

actual = broken_total(49900, 2)
assert isinstance(actual, int)
print("Type-only check passed:", actual)


**Change and explain:** Would executing this line tell coverage whether the amount is correct?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The wrong amount 49902 is still an integer.

</details>

### 3. Strengthen the claim

An exact requirement-derived expectation gives execution a meaning. This teaching catch exposes the deliberate failure without ending the notebook.

**Before running:** Predict the result or state change.

In [ ]:
try:
    assert actual == 99800
except AssertionError:
    print("The amount assertion detects:", actual, "!=", 99800)
print("Correct operation:", calculate_line_total(49900, 2))


**Change and explain:** Explain what the coverage report could establish without this comparison.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The amount check detects addition where multiplication was required.

</details>

### 4. Choose boundaries for their meaning

Nearby values can take different allowed and rejected paths. A free product and an empty quantity are different facts.

**Before running:** Predict the result or state change.

In [ ]:
import pytest
assert calculate_line_total(0, 2) == 0
for quantity in [0, -1, True]:
    with pytest.raises(ValueError):
        calculate_line_total(49900, quantity)
    print("Checked rejection:", repr(quantity))


**Change and explain:** Choose one additional input and explain the defect it could expose.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Zero price is accepted; these invalid quantities are rejected.

</details>

### 5. Supply an isolated source for real coverage

Coverage needs a real file to report locations. This visible temporary calculator uses the same rule as the notebook function.

**Before running:** Predict the result or state change.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess, sys

directory = TemporaryDirectory()
project = Path(directory.name)
pricing_source = """def calculate_line_total(price_paise, quantity):
    if type(quantity) is not int or quantity <= 0:
        raise ValueError("Quantity must be a positive integer")
    return price_paise * quantity
"""
print(pricing_source)


**Change and explain:** The temporary copy is separate from Nano-Zon and the notebook function.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The source contains an accepted path and a rejected path.

</details>

### 6. Supply tests that execute both paths but miss arithmetic

A suite can visit both outcomes yet check an insufficient success property.

**Before running:** Predict the result or state change.

In [ ]:
weak_tests = """import pytest
from pricing import calculate_line_total

def test_result_type():
    assert isinstance(calculate_line_total(49900, 2), int)

def test_rejected_quantity():
    with pytest.raises(ValueError):
        calculate_line_total(49900, 0)
"""
print(weak_tests)


**Change and explain:** Predict whether these assertions catch multiplication changed to addition.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Rejection is checked specifically, but the accepted amount is checked only by type.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Make the deliberate defect only in the temporary copy

A controlled mutation reveals which promises the tests actually establish. Keep the reference project intact.

**Before running:** Predict the result or state change.

In [ ]:
broken_source = pricing_source.replace("price_paise * quantity", "price_paise + quantity")
(project / "pricing.py").write_text(broken_source, encoding="utf-8")
(project / "test_pricing.py").write_text(weak_tests, encoding="utf-8")
print((project / "pricing.py").read_text(encoding="utf-8"))


**Change and explain:** Predict test status and coverage before the next cell.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Only the temporary calculator has the addition defect.

</details>

### 8. Run actual tests and coverage

Execution coverage and assertion strength answer different questions. The runner preserves the process result and diagnostic output.

**Before running:** Predict the result or state change.

In [ ]:
def run_checks():
    result = subprocess.run([
        sys.executable, "-m", "pytest", "test_pricing.py", "-q", "-p", "no:cacheprovider",
        "--cov=pricing", "--cov-branch", "--cov-report=term-missing",
        "--cov-report=xml:coverage.xml", "--junitxml=tests.xml",
    ], cwd=project, capture_output=True, text=True, check=False)
    print(result.stdout)
    print(result.stderr)
    print("Exit status:", result.returncode)
    return result.returncode

weak_status = run_checks()


**Change and explain:** Inspect the report even if coverage is complete. Which requirement is still unchecked?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

With the supplied suite, the arithmetic defect is expected to remain green despite complete measured execution. Read the actual report rather than assuming a percentage.

</details>

### 9. Add evidence that rejects the defect

Add a meaningful comparison rather than raising a percentage target. Failed runs should still retain their reports.

**Before running:** Predict the result or state change.

In [ ]:
strong_tests = weak_tests + """
def test_two_cables_cost_99800():
    assert calculate_line_total(49900, 2) == 99800
"""
(project / "test_pricing.py").write_text(strong_tests, encoding="utf-8")
strong_status = run_checks()
print("Reports retained:", (project / "coverage.xml").exists(), (project / "tests.xml").exists())


**Change and explain:** Why is this expected failing run useful evidence?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The new amount assertion is expected to fail with actual 49902; reports remain available.

</details>

### 10. Repair the behavior and repeat the same checks

A repeatable command allows a changed implementation to be compared with the same behavioral expectations.

**Before running:** Predict the result or state change.

In [ ]:
(project / "pricing.py").write_text(pricing_source, encoding="utf-8")
# Remove the temporary bytecode cache so rapid same-size edits cannot reuse it.
import shutil
shutil.rmtree(project / "__pycache__", ignore_errors=True)
repaired_status = run_checks()
print("Weak, strong, repaired:", weak_status, strong_status, repaired_status)


**Change and explain:** State exactly which defects the strengthened suite still does not check.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The expected exit sequence is zero, nonzero, zero. It is teaching intent, not captured execution output.

</details>

### 11. Read retained evidence and variable identity correctly

Reports retain what ran. Variable identities should be checked by relationships and shape, not by assuming a particular generated value.

**Before running:** Predict the result or state change.

In [ ]:
import xml.etree.ElementTree as ET
from uuid import uuid4
coverage = ET.parse(project / "coverage.xml").getroot()
print("Line rate:", coverage.attrib["line-rate"])
print("Branch rate:", coverage.attrib["branch-rate"])
order_id = uuid4().hex
saved = {"order_id": order_id}
assert saved["order_id"] == order_id
print("Order ID length:", len(order_id))


**Change and explain:** Explain why exact totals need exact assertions while generated IDs need a different check.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The report contains measured rates; the generated identity stays related to the returned record.

</details>

### 12. Finish the experiment and hand off automation

CI repeats declared installation and checks in another environment. This notebook cannot activate a remote workflow.

**Before running:** Predict the result or state change.

In [ ]:
print("Retained report files:", sorted(path.name for path in project.glob("*.xml")))
directory.cleanup()
print("Temporary project removed:", not project.exists())


**Change and explain:** Read the linked CI template: match dependencies, working directory, runner, exit status, and report upload after failure.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Only temporary files are removed. The supplied CI file remains a template until deliberately installed.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M9 lab guide](../../projects/nano-zon/labs/m09/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

### Apply the checks to project files

The [M9 source guide](../../teaching-snippets/m09/README.md) supplies the formal test/runner commands. The notebook is a small exploration; the Nano-Zon lab checks the complete application.

### CI is a separate configuration artifact

Read the [supplied workflow template](../../teaching-snippets/m09/s04_ci.yml). Identify installation, the shared runner, the working directory, failure exit status, and report upload after failure. The notebook does not activate remote CI or configure required checks.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.